# Task 7: Panorama-Style Warp (Homography)

Two overlapping views of the same scene; ORB matches then warp one image to the other.

## Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

full = cv2.imread('../CVLab02/assets/batman.jpg')
h, w = full.shape[:2]
# Overlapping crops simulate two handheld shots
split = int(w * 0.55)
img1 = full[:, :split + int(w * 0.15)]   # left + overlap
img2 = full[:, split - int(w * 0.15):]   # right + overlap


## ORB + Homography

In [ ]:
g1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
g2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
orb = cv2.ORB_create(800)
k1, d1 = orb.detectAndCompute(g1, None)
k2, d2 = orb.detectAndCompute(g2, None)
bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
matches = bf.match(d1, d2)
matches = sorted(matches, key=lambda m: m.distance)[:80]

if len(matches) < 4:
    raise RuntimeError('Not enough matches; try another image pair.')

pts1 = np.float32([k1[m.queryIdx].pt for m in matches])
pts2 = np.float32([k2[m.trainIdx].pt for m in matches])
H, _ = cv2.findHomography(pts2, pts1, cv2.RANSAC, 5.0)

tw = img1.shape[1] + img2.shape[1] // 2
th = max(img1.shape[0], img2.shape[0])
warp2 = cv2.warpPerspective(img2, H, (tw, th))
stitch = warp2.copy()
stitch[0:img1.shape[0], 0:img1.shape[1]] = img1

plt.figure(figsize=(12, 4))
plt.imshow(cv2.cvtColor(stitch, cv2.COLOR_BGR2RGB)); plt.title('Simple stitch (left + warped right)'); plt.axis('off')
plt.tight_layout(); plt.show()
